In [0]:
import time
import requests

from datetime import datetime, timezone

from pyspark.sql import Row
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    TimestampType
)

In [0]:
dbutils.widgets.text("city", "")
dbutils.widgets.text("latitude", "")
dbutils.widgets.text("longitude", "")
dbutils.widgets.text("pipeline_run_id", "")

city = dbutils.widgets.get("city").strip()
latitude = float(dbutils.widgets.get("latitude"))
longitude = float(dbutils.widgets.get("longitude"))
pipeline_run_id = dbutils.widgets.get("pipeline_run_id").strip()

In [0]:
if not city:
    raise ValueError("Parameter 'city' is required.")

if not pipeline_run_id:
    raise ValueError("Parameter 'pipeline_run_id' is required.")

if not (-90 <= latitude <= 90):
    raise ValueError(
        f"Invalid latitude: {latitude}"
    )

if not (-180 <= longitude <= 180):
    raise ValueError(
        f"Invalid longitude: {longitude}"
    )

In [0]:
API_URL = "https://api.open-meteo.com/v1/forecast"

RAW_BASE_PATH = (
    "abfss://raw@stweatherdefaultstorage.dfs.core.windows.net/"
    "weather/"
)

MAX_RETRIES = 3
REQUEST_TIMEOUT_SECONDS = 30

In [0]:
def fetch_weather(
    latitude: float,
    longitude: float,
    max_retries: int = 3
):

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "current": (
            "temperature_2m,"
            "relative_humidity_2m,"
            "wind_speed_10m,"
            "precipitation"
        ),
        "timezone": "UTC"
    }

    last_error = None

    for attempt in range(1, max_retries + 1):

        try:

            response = requests.get(
                API_URL,
                params=params,
                timeout=REQUEST_TIMEOUT_SECONDS
            )

            response.raise_for_status()

            return response.json()

        except requests.RequestException as exc:

            last_error = exc

            if attempt == max_retries:
                break

            sleep_seconds = 2 ** (attempt - 1)

            print(
                f"API attempt {attempt} failed. "
                f"Retrying in {sleep_seconds}s..."
            )

            time.sleep(sleep_seconds)

    raise RuntimeError(
        f"Weather API failed after "
        f"{max_retries} attempts: {last_error}"
    )

In [0]:
api_response = fetch_weather(
    latitude=latitude,
    longitude=longitude,
    max_retries=MAX_RETRIES
)

if "current" not in api_response:
    raise ValueError(
        "API response does not contain 'current'."
    )

current = api_response["current"]

In [0]:
ingestion_time = datetime.now(timezone.utc)

In [0]:
raw_schema = StructType(
    [
        StructField("city", StringType(), False),
        StructField("latitude", DoubleType(), False),
        StructField("longitude", DoubleType(), False),
        StructField("event_time", StringType(), True),
        StructField("temperature_c", DoubleType(), True),
        StructField("humidity_pct", DoubleType(), True),
        StructField("wind_speed_kmh", DoubleType(), True),
        StructField("precipitation_mm", DoubleType(), True),
        StructField("ingestion_time", TimestampType(), False),
        StructField("source", StringType(), False),
        StructField("pipeline_run_id", StringType(), False),
    ]
)

In [0]:
record = (
    city,
    latitude,
    longitude,
    current.get("time"),
    current.get("temperature_2m"),
    current.get("relative_humidity_2m"),
    current.get("wind_speed_10m"),
    current.get("precipitation"),
    ingestion_time,
    "open-meteo",
    pipeline_run_id,
)

In [0]:
raw_df = spark.createDataFrame([record], schema=raw_schema)

In [0]:
safe_city = city.lower().strip().replace(" ", "_").replace("/", "_")

event_date = ingestion_time.strftime("%Y-%m-%d")

raw_path = f"{RAW_BASE_PATH}city={safe_city}/date={event_date}/"

In [0]:
(raw_df.write.format("parquet").mode("append").save(raw_path))

In [0]:
print("=" * 60)
print("WEATHER INGESTION SUCCESS")
print("=" * 60)
print(f"City            : {city}")
print(f"Latitude        : {latitude}")
print(f"Longitude       : {longitude}")
print(f"Pipeline Run ID : {pipeline_run_id}")
print(f"Ingestion Time  : {ingestion_time}")
print(f"RAW Path        : {raw_path}")
print("=" * 60)